# TaskB: Matching the JobTtile with the Required SkillSet

In this notebook, we prepare the shared-track Task-B dataset 
- for developing the job-title to Skills matching model and
- for evaluating the predicted matched skills using evaluation script [task's evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script).

- Additionally, the provided format is also compatible with the testset.

## Imports

- Basics

In [ ]:
import os
import sys

- Medium

In [ ]:
import pandas as pd
import numpy as np
import json
import subprocess
import ast

- Advanced

In [ ]:
import networkx as nx

from sentence_transformers import SentenceTransformer, util
from codecarbon import EmissionsTracker

## Path preparation (root, data, models, results, and evaluations)

In [ ]:
root_data_path="taskB"

In [ ]:
#Training data path
taskB_training_path = os.path.join(root_data_path, "training")

#validation data path
validation_data_path = os.path.join(root_data_path, "validation")

#testing data path
test_data_path = os.path.join(root_data_path, "test")

In [ ]:
models_path = os.path.join(root_path, "models")
results_path = os.path.join(root_path, "results")
evaluations_path = os.path.join(root_path, "evaluations")

In [ ]:
sbert_model="all-MiniLM-L6-v2"
sentence_transformer_model=os.path.join(models_path, sbert_model)

## Training Data (Preparation and Modeling)

- **Reading job2skill file**

In [ ]:
# Read job2skill file
job2skill = pd.read_csv(os.path.join(training_data_path, 'job2skill.tsv'),
                        sep="\t",
                        names=["job_id","skill_id","rel_type"])
job2skill.head()

- **Reading Job2Terms**

In [ ]:
# Read json files
jobid2terms_path = os.path.join(training_data_path, "jobid2terms.json")
with open(jobid2terms_path, 'r') as file:
    jobid2terms = json.load(file)
#print (jobid2terms['http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34'])

In [ ]:
for jobid in jobid2terms:
    print (jobid)
    print (jobid2terms[jobid])
    break

- **Reading Skilled2Terms**

In [ ]:
# Read json files
skillid2terms_path = os.path.join(training_data_path, "skillid2terms.json")
with open(skillid2terms_path, 'r') as file:
    skillid2terms = json.load(file)
#print (skillid2terms)

In [ ]:
for skillid in skillid2terms:
    print (skillid)
    print (skillid2terms[skillid])
    break

## Validation data (Preparataion and Prediction)

#### Load queries and corpus elements in English from the Validation folder:

In [ ]:
dev_queries_path = os.path.join(validation_data_path, "queries")
dev_corpus_elements_path = os.path.join(validation_data_path, "corpus_elements")

In [ ]:
dev_queries = pd.read_csv(dev_queries_path,sep="\t")
dev_corpus_elements = pd.read_csv(dev_corpus_elements_path, sep="\t")

In [ ]:
len(dev_queries), len(dev_corpus_elements)

#### Transform `skill_aliases` column to a list of strings:

In [ ]:
dev_corpus_elements["skill_aliases"] = dev_corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))

#### Approaches of Skill term expansion
 

##### **Approach A (doc_A): Extracting skill related terms**

In [ ]:
def related_skill_terms(terms_list):
    return " ".join(terms_list)

In [ ]:
dev_corpus_elements["doc_A"] = dev_corpus_elements["skill_aliases"].apply(lambda x: related_skill_terms(x))

In [ ]:
print(len(dev_corpus_elements))
dev_corpus_elements.columns

##### **Approach B (doc_B): Extracting terms of jobid associated with the skill**

In [ ]:
def get_essential_jobids(skill_id):
    return list(job2skill[(job2skill["skill_id"]==skill_id) & (job2skill["rel_type"]=="essential")]["job_id"])
    

In [ ]:
def related_jobid_terms(skill_id):
    related_essential_jobids = get_essential_jobids(skill_id)
    terms_B = []
    for related_jobid in related_essential_jobids:
        related_jobid_terms = jobid2terms[related_jobid]
        terms_B.extend(related_jobid_terms)
    return " ".join(terms_B)

In [ ]:
dev_corpus_elements["doc_B"] = dev_corpus_elements["esco_uri"].apply(lambda x: related_jobid_terms(x))

In [ ]:
print (len(dev_corpus_elements))
dev_corpus_elements.columns

##### **Approach C (doc_C): Extracting terms of related skill of related jobid**
  - Skill_id -> {J_1, J_2, ..., J_N} :: J_1-> {S_1, S_2, ..., S_M} :: S_1 -> Terms

In [ ]:
def get_essential_skillids(job_id):
    return list(job2skill[(job2skill["job_id"]==job_id) & (job2skill["rel_type"]=="essential")]["skill_id"])       

In [ ]:
def related_job2skill_terms(skill_id):
    related_essential_jobids = get_essential_jobids(skill_id)
    
    related_skillids = []
    for related_jobid in related_essential_jobids:
        skill_ids = get_essential_skillids(related_jobid)
        skill_ids.remove(skill_id)
        related_skillids.extend(skill_ids)
    
    terms_C = []
    for related_skillid in related_skillids:
        related_skillid_terms = skillid2terms[related_skillid]
        terms_C.extend(related_skillid_terms)

    return " ".join(terms_C)

In [ ]:
dev_corpus_elements["doc_C"] = dev_corpus_elements["esco_uri"].apply(lambda x: related_job2skill_terms(x))

In [ ]:
print (len(dev_corpus_elements))
dev_corpus_elements.columns

#### Load simple embedding model:

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2", token=False)

In [ ]:
tracker = EmissionsTracker()
tracker.start_task("all-MiniLM-L6-v2")

##### Generate a mapping dictionary between IDs and texts from query

In [ ]:
dev_queries_ids = dev_queries.q_id.to_list()
dev_queries_texts = dev_queries.jobtitle.to_list()
dev_queries_map = dict(zip(dev_queries_ids, dev_queries_texts))

#### Creating a mapping dictionary of corpus element texts from each of the approaches to corpus_ids

In [ ]:
corpus_ids = dev_corpus_elements.c_id.to_list()
corpus_esco_uri = dev_corpus_elements.esco_uri.to_list()
print (len(corpus_ids), len(corpus_esco_uri))

##### Approaches A, B, and C

In [ ]:
#doc_A
corpus_doc_A_texts = dev_corpus_elements.doc_A.to_list()
map_corpus_doc_A = dict(zip(corpus_ids, corpus_doc_A_texts))
print (len(corpus_doc_A_texts), len(corpus_ids), len(map_corpus_doc_A))

#doc_B
corpus_doc_B_texts = dev_corpus_elements.doc_B.to_list()
map_corpus_doc_B = dict(zip(corpus_ids, corpus_doc_B_texts))
print (len(corpus_doc_B_texts), len(corpus_ids), len(map_corpus_doc_B))

#doc_C
corpus_doc_C_texts = dev_corpus_elements.doc_C.to_list()
map_corpus_doc_C = dict(zip(corpus_ids, corpus_doc_C_texts))
print (len(corpus_doc_C_texts), len(corpus_ids), len(map_corpus_doc_C))

In [ ]:
len(corpus_ids), len(corpus_esco_uri), len(corpus_doc_A_texts), len(map_corpus_doc_A), len(corpus_doc_B_texts), len(map_corpus_doc_B), len(corpus_doc_C_texts), len(map_corpus_doc_C)

- Encode queries and corpus elements:

In [ ]:
query_embeddings = model.encode(dev_queries_texts, convert_to_tensor=True)
#corpus_embeddings = model.encode(corpus_doc_A_texts, convert_to_tensor=True)

In [ ]:
corpus_doc_A_embedding = model.encode(corpus_doc_A_texts, convert_to_tensor=True)
corpus_doc_B_embedding = model.encode(corpus_doc_B_texts, convert_to_tensor=True)
corpus_doc_C_embedding = model.encode(corpus_doc_C_texts, convert_to_tensor=True)

In [ ]:
corpus_doc_A_embedding.shape,corpus_doc_B_embedding.shape,corpus_doc_C_embedding.shape 

Compute similarities

In [ ]:
similarities_query_doc_A = util.cos_sim(query_embeddings, corpus_doc_A_embedding).cpu().numpy()

similarities_query_doc_B = util.cos_sim(query_embeddings, corpus_doc_B_embedding).cpu().numpy()

similarities_query_doc_C = util.cos_sim(query_embeddings, corpus_doc_C_embedding).cpu().numpy()

emissions = tracker.stop_task("all-MiniLM-L6-v2")

In [ ]:
similarities_query_doc_A.shape, similarities_query_doc_B.shape, similarities_query_doc_C.shape

In [ ]:
len(similarities_query_doc_A[0])

In [ ]:
similarities_query_doc_AB = similarities_query_doc_A + similarities_query_doc_B
similarities_query_doc_AC = similarities_query_doc_A + similarities_query_doc_C
similarities_query_doc_BC = similarities_query_doc_B + similarities_query_doc_C
similarities_query_doc_ABC = similarities_query_doc_A + similarities_query_doc_B + similarities_query_doc_C
similarities_query_doc_WABC = similarities_query_doc_A*0.5 + similarities_query_doc_B*0.3 + similarities_query_doc_C*0.2

## Prepare submission file

The submissions must follow the TREC Run File format, including headers in the output file. This means that the fle have 6 space-spearated columns per line, with following information:

- q_id: Query ID.
- Q0: A constant identifier, usually "Q0".
- doc_id: ID of the retrieved document.
- rank: Position of the document in the ranking.
- score: Relevance score assigned by the model.
- tag: Experiment name

In [ ]:
import numpy as np

def get_ranked_result_list(similarities_query_doc, corpus_doc):
    results = []
    results_name = []
    
    for q_idx, q_id in enumerate(dev_queries_ids):
        sorted_indices = np.argsort(-similarities_query_doc[q_idx])
        used_doc_ids = set()
        rank_counter = 0
        for c_idx in sorted_indices:  # Consider the full list.
            doc_id = corpus_ids[c_idx]
            # If doc_id was already processed, go to the next one.
            if doc_id in used_doc_ids:
                continue
            used_doc_ids.add(doc_id)
            rank_counter += 1
    
            query_name = dev_queries_map[q_id]
            doc_name = corpus_doc[c_idx]
            score = similarities_query_doc[q_idx, c_idx]
    
            results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} A_model")
            results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} A_model")

    return results, results_name

The list has this structure

Let's save the list as a file:

In [ ]:
results_A, results_A_name = get_ranked_result_list(similarities_query_doc_A, corpus_doc_A_texts)
results_B, results_B_name = get_ranked_result_list(similarities_query_doc_B, corpus_doc_B_texts)
results_C, results_C_name = get_ranked_result_list(similarities_query_doc_C, corpus_doc_C_texts)
results_AB, results_AB_name = get_ranked_result_list(similarities_query_doc_AB, corpus_doc_A_texts)
results_AC, results_AC_name = get_ranked_result_list(similarities_query_doc_AC, corpus_doc_A_texts)
results_BC, results_BC_name = get_ranked_result_list(similarities_query_doc_BC, corpus_doc_B_texts)
results_ABC, results_ABC_name = get_ranked_result_list(similarities_query_doc_ABC, corpus_doc_A_texts)
results_WABC, results_WABC_name = get_ranked_result_list(similarities_query_doc_WABC, corpus_doc_A_texts)

result_approach_A_taskB = os.path.join(results_path, "evaluation_approach_A_taskB.trec")
with open(result_approach_A_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_A))

result_approach_B_taskB = os.path.join(results_path, "evaluation_approach_B_taskB.trec")
with open(result_approach_B_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_B))

result_approach_C_taskB = os.path.join(results_path, "evaluation_approach_C_taskB.trec")
with open(result_approach_C_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_C))

result_approach_AB_taskB = os.path.join(results_path, "evaluation_approach_AB_taskB.trec")
with open(result_approach_AB_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_AB))

result_approach_AC_taskB = os.path.join(results_path, "evaluation_approach_AC_taskB.trec")
with open(result_approach_AC_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_AC))

result_approach_BC_taskB = os.path.join(results_path, "evaluation_approach_BC_taskB.trec")
with open(result_approach_BC_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_BC))

result_approach_ABC_taskB = os.path.join(results_path, "evaluation_approach_ABC_taskB.trec")
with open(result_approach_ABC_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_ABC))

result_approach_WABC_taskB = os.path.join(results_path, "evaluation_approach_WABC_taskB.trec")
with open(result_approach_WABC_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_WABC))

emissions_path = os.path.join(evaluations_path, "emissions.json")
json.dump(dict(emissions.values), open(emissions_path, "w"), ensure_ascii=False, indent=4)

## Evaluation

For the evaluation, we will use the official [TalentCLEF evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script), which uses the Ranx library under the hood.

First, clone the repo and install the requirements file:

In [ ]:
!git clone https://github.com/TalentCLEF/talentclef25_evaluation_script.git
!pip install -r /content/talentclef25_evaluation_script/requirements.txt


Then, select the Qrels file and the Run file to perform the evaluation.


In [ ]:
qrels_file = os.path.join(root_path, "validation", "qrels.tsv")

run_baseline_file = os.path.join(results_path, "evaluation_baseline_taskB.trec")

run_A_file = os.path.join(results_path, "evaluation_approach_A_taskB.trec")

run_B_file = os.path.join(results_path, "evaluation_approach_B_taskB.trec")

run_C_file = os.path.join(results_path, "evaluation_approach_C_taskB.trec")

run_AB_file = os.path.join(results_path, "evaluation_approach_AB_taskB.trec")

run_AC_file = os.path.join(results_path, "evaluation_approach_AC_taskB.trec")

run_BC_file = os.path.join(results_path, "evaluation_approach_BC_taskB.trec")

run_ABC_file = os.path.join(results_path, "evaluation_approach_ABC_taskB.trec")

run_WABC_file = os.path.join(results_path, "evaluation_approach_WABC_taskB.trec")

In [ ]:
print ("Baseline")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_baseline_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach A")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_A_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach B")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_B_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach C")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_C_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach AB")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_AB_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach AC")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_AC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach BC")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_BC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach ABC")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_ABC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach WABC")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_WABC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

In [ ]:
run_baseline_file = os.path.join(results_path, "evaluation_baseline_taskB.trec")

In [ ]:
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

In [ ]:
!python talentclef25_evaluation_script/talentclef_evaluate.py --qrels ~/Documents/projects/data/ir_data/talent_clef/TaskB/validation/qrels.tsv --run ~/Documents/projects/data/ir_data/talent_clef/TaskB/results/evaluation_baseline_taskB.tre

In [ ]:
#model.save(taskB_models_path)

### Testing data (Preparation and Prediction)

In [ ]:
test_queries_path = os.path.join(test_data_path, "queries")
test_corpus_elements_path = os.path.join(test_data_path, "corpus_elements")

In [ ]:
test_queries = pd.read_csv(test_queries_path,sep="\t")
test_corpus_elements = pd.read_csv(test_corpus_elements_path, sep="\t")

In [ ]:
test_corpus_elements["skill_aliases"] = test_corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))

In [ ]:
test_corpus_elements.shape

In [ ]:
test_queries_ids = test_queries.q_id.to_list()
test_queries_texts = test_queries.jobtitle.to_list()
test_map_queries = dict(zip(test_queries_ids, test_queries_texts))

In [ ]:
len(test_map_queries)

In [ ]:
test_list_aliases_df = test_corpus_elements.explode("skill_aliases")

In [ ]:
test_corpus_ids = test_list_aliases_df.c_id.to_list()
test_corpus_esco_uri = test_list_aliases_df.esco_uri.to_list()
test_corpus_texts = test_list_aliases_df.skill_aliases.to_list()
test_map_corpus = dict(zip(test_corpus_texts, test_corpus_ids))

In [ ]:
len(test_corpus_ids), len(test_corpus_esco_uri), len(test_corpus_texts), len(test_map_corpus)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2", token=False)

In [ ]:
test_query_embeddings = model.encode(test_queries_texts, convert_to_tensor=True)
test_corpus_embeddings = model.encode(test_corpus_texts, convert_to_tensor=True)

In [ ]:
test_similarities = util.cos_sim(test_query_embeddings, test_corpus_embeddings).cpu().numpy()


In [ ]:
test_similarities.shape

In [ ]:
import numpy as np
results = []
results_name = []

for q_idx, q_id in enumerate(test_queries_ids):
    sorted_indices = np.argsort(-test_similarities[q_idx])
    used_doc_ids = set()
    rank_counter = 0
    for c_idx in sorted_indices:  # Consider the full list.
        doc_id = test_corpus_ids[c_idx]
        # If doc_id was already processed, go to the next one.
        if doc_id in used_doc_ids:
            continue
        used_doc_ids.add(doc_id)
        rank_counter += 1

        query_name = test_map_queries[q_id]
        doc_name = test_corpus_texts[c_idx]
        score = test_similarities[q_idx, c_idx]

        results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} baseline_model")
        results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} baseline_model")

In [ ]:
result_baseline_taskB = os.path.join(results_path, "test_baseline_taskB.trec")
with open(result_baseline_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results))